## Use MCP server using OpenAI SDK Agent

In this demo, we will build an Agent using OpenAI SDK Agent framework. 

We will configure the agent to use few community MCP server. 

With this demo, you will be learn how to:
1. Build an Agent using OpenAI SDK Agent framework.
2. Configure the Agent to use a community MCP server.
3. Run the Agent and interact with it.


#### Community MCP server

Example MCP servers can be found at https://modelcontextprotocol.io/examples

We will explore 2 community mcp server, one using python (uvx) and other using node (npx) so that you can learn both python and node based mcp server

1. Time - Time and timezone conversion capabilities (https://github.com/modelcontextprotocol/servers/tree/main/src/time)
2. Filesystem - Secure file operations with configurable access controls (https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem)

You can also try Fetch mcp server: Fetch - Web content fetching and conversion for efficient LLM usage (https://github.com/modelcontextprotocol/servers/tree/main/src/fetch)



#### Import libraries

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, SQLiteSession
from agents.mcp import MCPServerStdio

#### Load Environment

In [ ]:
load_dotenv(override=True)

#### Lets use Time MCP server
Check available tools in the MCP server

In [ ]:
#### Check available tools
fetch_params = {"command": "uvx", "args": ["mcp-server-time"]}

async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60) as server:
    tool_lists = await server.list_tools()

tool_lists

#### Build Agent with Time MCP server

Here we are using MCPServerStdio (stdio mode).

This will use uvx to fetch, install (if needed), and run the mcp-server-time package.

If mcp-server-time is not already installed, uvx will download it from GitHub (or the registry), install it in a local cache, and then start the MCP server as a subprocess.

MCPServerStdio then connects to this subprocess via standard input/output.



In [ ]:
model = "gpt-4.1-nano"

agent_instructions = (
        "You are a helpful assistant that can use tools to answer questions. "
    )

async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60) as server:
    general_agent = Agent(name="general_agent",
                    instructions=agent_instructions,
                    model=model,
                    mcp_servers=[server])
    query="what is the current time in Chennai, India?"
    result = await Runner.run(general_agent, query)
    print(result.final_output)
    

#### Create a chatbot with the Agent

In [ ]:
session = SQLiteSession("chat_session")

async def chat(message, history):
    async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60) as server:
        general_agent = Agent(name="general_agent",
                        instructions=agent_instructions,
                        model=model,
                        mcp_servers=[server])
        result = await Runner.run(general_agent, message)
        return result.final_output


import gradio as gr
gr.ChatInterface(
    chat,
    title="Time Zone Chatbot",
    description="Chat with the Time Zone Expert"
).launch()

#### Lets use File System MCP server
Check available tools in the MCP server. Node based

In [ ]:
fetch_params = {"command": "npx", "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        "/mnt/c/Users/PriyabrataPanigrahi/Downloads/ai/prpanigrahi/git",
        "/mnt/c/Users/PriyabrataPanigrahi/Downloads/ai/prpanigrahi/git/mcp_for_bioinformatics_course"
      ]}

async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60) as server:
    tool_lists = await server.list_tools()

tool_lists

#### Build Agent with chatbot interface

In [ ]:
session = SQLiteSession("chat_session")
async def chat(message, history):
    async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60) as server:
        general_agent = Agent(name="general_agent",
                        instructions=agent_instructions,
                        model=model,
                        mcp_servers=[server])
        result = await Runner.run(general_agent, message)
        return result.final_output


import gradio as gr
gr.ChatInterface(
    chat,
    title="File system Chatbot",
    description="Chat with the File System Expert"
).launch()